In [1]:
!pip install -q -U datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 19.2 MB/s eta 0:00:00


In [4]:
from datasets import load_dataset
train = load_dataset("risqaliyevds/uzbek_ner")["train"]
CLASSES = ["PERSON", "ORG", "GPE", "LOC", "DATE"] # fixed priority order
SOURCE_KEYS = {
"PERSON": ["PERSON", "PER"], # Phase 1: pure union, 0 overlapping rows
"ORG": ["ORG"],
"GPE": ["GPE"],
"LOC": ["LOC"],
"DATE": ["DATE"], # PERIOD deliberately absent (8-string artifact)
}
print(len(train), "rows;", len(CLASSES), "classes")

README.md:   0%|          | 0.00/3.05k [00:00<?, ?B/s]

uzbek_ner.json:   0%|          | 0.00/24.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/19609 [00:00<?, ? examples/s]

19609 rows; 5 classes


In [6]:
APOS = ["\u2018", "\u2019", "\u02bb", "\u02bc", "`", "\u00b4"]
def normalize(s):
    for ch in APOS:
        s = s.replace(ch, "'")
    return s
# quick self-test
sample = "O\u2018zbekiston va o\u02bbsish"
print(repr(sample), "->", repr(normalize(sample)))

'O‘zbekiston va oʻsish' -> "O'zbekiston va o'sish"


In [8]:
PUNCT = ".',;:!?()[]{}\"\'«»—–…"
def strip_edges(w):
    return w.strip(PUNCT)
t = normalize("«Spendrups» kompaniyasi, Stokholmdagi.")
words = t.split()
print(words)
print([strip_edges(w) for w in words])

['«Spendrups»', 'kompaniyasi,', 'Stokholmdagi.']
['Spendrups', 'kompaniyasi', 'Stokholmdagi']


In [10]:
def collect_entities(row):
    out = {}
    for cls in CLASSES:
        raw = []
        for key in SOURCE_KEYS[cls]:
            v = row["ner"].get(key)
            if v:
                raw += v
        cleaned = []
        for e in raw:
            if not isinstance(e, str):
                continue
            e = normalize(e).strip()
            if len(e) >= 2 and len(e.split()) <= 6:
                cleaned.append(e)
        uniq = sorted(set(cleaned),
                      key=lambda e: (len(e.split()), len(e)),
                      reverse=True)
        if uniq:
            out[cls] = uniq
    return out
# self-test on row 0
ents0 = collect_entities(train[0])
for cls, lst in ents0.items():
    print(cls, "->", lst)

PERSON -> ['Shvetsiya bosh vaziri Stefan Lyoven']
ORG -> ['Spendrups kompaniyasi']
GPE -> ['Shvetsiya bosh vaziri Stefan Lyoven', "O'zbekiston", 'Shvetsiya']
LOC -> ['Drottninggatanda', 'Stokholmdagi']


In [12]:
def to_bio(text, ents):
    words = normalize(text).split()
    core = [strip_edges(w) for w in words] # matching forms
    tags = ["O"] * len(words)
    misses = []
    for cls in CLASSES: # fixed priority
        for ent in ents.get(cls, []): # longest first (Step 3)
            ew = [strip_edges(x) for x in ent.split()]
            ew = [x for x in ew if x]
            if not ew:
                continue
            n = len(ew)
            found = False
            for i in range(len(words) - n + 1):
                if any(t != "O" for t in tags[i:i+n]):
                    continue # never overwrite
                head_ok = all(core[i+j] == ew[j] for j in range(n - 1))
                tail_ok = core[i+n-1].startswith(ew[-1])
                if head_ok and tail_ok:
                    tags[i] = "B-" + cls
                    for j in range(1, n):
                        tags[i+j] = "I-" + cls
                    found = True # keep scanning: tag ALL occurrences
            if not found:
                misses.append((cls, ent))
    return words, tags, misses

In [14]:
words, tags, misses = to_bio(train[0]["text"],
                         collect_entities(train[0]))
for w, t in zip(words, tags):
    print(f"{w:20} {t}")
print()
print("MISSES:", misses)

Shvetsiya            B-GPE
hukumati             O
Stokholmdagi         B-LOC
asosiy               O
piyodalar            O
ko'chasi             O
Drottninggatanda     B-LOC
odamlar              O
ustiga               O
yuk                  O
mashinasini          O
haydab               O
borgani              O
gumon                O
qilinayotgan         O
shaxsni              O
qo'lga               O
oldi.                O
Bu                   O
haqda                O
Expressen            O
TV                   O
efirida              O
Shvetsiya            B-PERSON
bosh                 I-PERSON
vaziri               I-PERSON
Stefan               I-PERSON
Lyoven               I-PERSON
ma'lum               O
qildi.               O
Ushbu                O
shaxs                O
surati               O
ijtimoiy             O
tarmoqda             O
tarqalmoqda.         O
U                    O
Spendrups            B-ORG
kompaniyasiga        I-ORG
tegishli             O
yuk                  O
ma

In [16]:
from collections import Counter
import pandas as pd

listed = Counter(); missed = Counter(); miss_examples = Counter()
tag_counter = Counter(); rows_with_span = 0
converted = []

for idx, row in enumerate(train):
    ents = collect_entities(row)
    words, tags, misses = to_bio(row["text"], ents)
    converted.append({"tokens": words, "tags": tags})

    for cls, lst in ents.items():
        listed[cls] += len(lst)

    for cls, ent in misses:
        missed[cls] += 1
        miss_examples[(cls, ent)] += 1

    tag_counter.update(tags)
    if any(t != "O" for t in tags):
        rows_with_span += 1

    if (idx + 1) % 5000 == 0:
        print(idx + 1, "rows done")

rep = pd.DataFrame({
    "listed": pd.Series(listed),
    "matched": pd.Series({c: listed[c] - missed[c] for c in listed}),
})
rep["match_rate_%"] = (rep["matched"] / rep["listed"] * 100).round(1)
rep["spans_tagged"] = pd.Series({c: tag_counter["B-" + c] for c in listed})

print(rep.loc[CLASSES].to_string())
total = sum(tag_counter.values())
print()
print(f"rows with at least one span: {rows_with_span} / {len(train)}")
print(f"O share of all tokens: {tag_counter['O'] / total * 100:.1f} %")
print()
print("Top unmatched entities:")
for (cls, ent), c in miss_examples.most_common(12):
    print(f" {c:4} {cls:7} {ent}")

5000 rows done
10000 rows done
15000 rows done
        listed  matched  match_rate_%  spans_tagged
PERSON   25006    24453          97.8         29747
ORG      27306    22299          81.7         31187
GPE      50031    31228          62.4         50108
LOC      38647    16404          42.4         21670
DATE     10901     9874          90.6         11011

rows with at least one span: 19445 / 19609
O share of all tokens: 86.5 %

Top unmatched entities:
 15289 GPE     O'zbekiston
 4814 LOC     Toshkent
 4382 LOC     O'zbekiston
  901 ORG     XDP
  642 ORG     O'zbekiston Respublikasi Madaniyat vazirligi
  616 ORG     O'zbekiston Respublikasi
  443 GPE     Toshkent
  415 LOC     Rossiya
  381 LOC     Suriya
  302 LOC     AQSh
  252 GPE     Rossiya
  232 LOC     Ukraina


In [17]:
from collections import Counter
import pandas as pd

listed2 = Counter(); missed2 = Counter()
absent = Counter(); present_missed = Counter()
examples_pm = {}

for row in train:
    ents = collect_entities(row)
    _, _, misses = to_bio(row["text"], ents)
    text_n = normalize(row["text"])
    for cls, lst in ents.items():
        listed2[cls] += len(lst)
    for cls, ent in misses:
        missed2[cls] += 1
        if ent in text_n:
            present_missed[cls] += 1
            ex = examples_pm.setdefault(cls, [])
            if len(ex) < 5:
                ex.append(ent)
        else:
            absent[cls] += 1

audit = pd.DataFrame({
    "listed": pd.Series(listed2),
    "matched": pd.Series({c: listed2[c] - missed2[c] for c in listed2}),
    "absent_from_text": pd.Series(absent),
    "present_but_missed": pd.Series(present_missed),
}).fillna(0).astype(int)
audit["raw_rate_%"] = (audit["matched"] / audit["listed"] * 100).round(1)
audit["adjusted_rate_%"] = (audit["matched"] /
                            (audit["listed"] - audit["absent_from_text"]) * 100).round(1)
print(audit.loc[CLASSES].to_string())
print()
for cls, ex in examples_pm.items():
    print(cls, "present-but-missed examples:", ex)

        listed  matched  absent_from_text  present_but_missed  raw_rate_%  adjusted_rate_%
PERSON   25006    24453               487                  66        97.8             99.7
ORG      27306    22299              3594                1413        81.7             94.0
GPE      50031    31228             15956                2847        62.4             91.6
LOC      38647    16404              8155               14088        42.4             53.8
DATE     10901     9874               724                 303        90.6             97.0

GPE present-but-missed examples: ['Shvetsiya bosh vaziri Stefan Lyoven', 'Xitoy', 'Shvetsiya', "O'zbekiston", "O'zbekiston"]
LOC present-but-missed examples: ['Suriya', 'Xitoy', 'Stokholm markazida', 'Xitoy', 'Suriya']
ORG present-but-missed examples: ["O'zsanoatqurilishbank", 'Xezbollah', "O'zbekneftegaz", 'Gazprom', "Dubay amirining o'g'li"]
DATE present-but-missed examples: ['2016 yil', 'mart kuni', '13 dekabr', '15 dekabr', '2022 yil']
PERSON pr

In [18]:
rows_dup = 0; dups = 0
for row in train:
    g = {normalize(x).strip() for x in (row["ner"].get("GPE") or []) if isinstance(x, str)}
    l = {normalize(x).strip() for x in (row["ner"].get("LOC") or []) if isinstance(x, str)}
    inter = (g & l) - {""}
    if inter:
        rows_dup += 1; dups += len(inter)
print(rows_dup, "rows list the same string under BOTH GPE and LOC;", dups, "shared strings")

7993 rows list the same string under BOTH GPE and LOC; 13072 shared strings


In [22]:
import random
random.seed(42)
for i in random.sample(range(len(converted)), 5):
    print(f"===== row {i} =====")
    for w, t in zip(converted[i]["tokens"], converted[i]["tags"]):
        mark = " <---" if t != "O" else ""
        print(f"{w:22} {t}{mark}")
    print()

===== row 3648 =====
XXR                    B-ORG <---
Xalq                   I-ORG <---
banki                  I-ORG <---
(markaziy              O
banki)                 O
mamlakat               O
milliy                 O
valyutasi              O
jenminbi               O
(yuanning              O
rasmiy                 O
nomi)ning              O
dollarga               O
nisbatan               O
kursini                O
153                    O
punktga                O
tushirdi.              O
Endilikda,             O
bir                    O
dollar                 O
6,6528                 O
yuanga                 O
tengdir.               O
Bu                     O
so'nggi                O
6                      O
yildagi                O
eng                    O
past                   O
ko'rsatkichdir.        O
China                  B-ORG <---
Securities             I-ORG <---
sarmoya                I-ORG <---
kompaniyasi            I-ORG <---
eksperti               O
Chjen           

In [24]:
def bio_violations(tags):
    v, prev = 0, "O"
    for t in tags:
        if t.startswith("I-") and prev not in ("B-" + t[2:], t):
            v += 1
        prev = t
    return v
bad_bio = sum(bio_violations(c["tags"]) for c in converted)
bad_len = sum(1 for c in converted if len(c["tokens"]) !=
                          len(c["tags"])) # Adjusted indentation for bad_len
empty = sum(1 for c in converted if len(c["tokens"]) == 0)
print("BIO violations:", bad_bio, "| length mismatches:", bad_len, "| empty rows:", empty)

BIO violations: 0 | length mismatches: 0 | empty rows: 0


In [25]:
from datasets import Dataset
import shutil
bio_ds = Dataset.from_list(converted)
print(bio_ds)
bio_ds.save_to_disk("uzbek_ner_bio")
rep.to_csv("phase2_conversion_report.csv")
shutil.make_archive("uzbek_ner_bio", "zip", "uzbek_ner_bio")
from google.colab import files
files.download("uzbek_ner_bio.zip")
files.download("phase2_conversion_report.csv")

Dataset({
    features: ['tokens', 'tags'],
    num_rows: 19609
})


Saving the dataset (0/1 shards):   0%|          | 0/19609 [00:00<?, ? examples/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>